# DNAStream tutorial
This tutorial will demonstrate how to:  
- Create a DNAStream HDF5 file
- Stream data from a DNAStream file
- how to use the context manager

## Create a DNAStream HDF5 file
`DNAStream` is an interface to an HDF5-file providing an opinionated and specified multi-modal data structure for organizing key measurements of DNA sequencing data and facilitating downstream evolutionary analysis. It provides compact on-disk storage, fast partial reads, and a structured way to track entities, links, and changes over time.`

The `DNAstream.create(...)` function initializes the HDF5-file and creates all built-in datasets.  It returns a `DNAStream` object that provides an interface to the HDF5 file. 

When finished streaming the data, `close()` is used to safely close the stream in order to avoid file corruption.

In [1]:
from pathlib import Path
import tempfile
from dnastream import DNAStream

tmpdir = Path(tempfile.mkdtemp(prefix="dnastream_tutorial_"))
myfile = tmpdir / "temp.h5"

# Create the DNAStream file (fails if it already exists)
ds = DNAStream.create(path=str(myfile), patient_id="patientX", verbose=True)
ds.close()

print("Created:", myfile)

2026-02-05 13:03:06 - INFO - Created DNAStream file /var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_oj7bmbg2/temp.h5
2026-02-05 13:03:06 - INFO - Connection to DNAStream file '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_oj7bmbg2/temp.h5' closed.


Created: /var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_oj7bmbg2/temp.h5


## Opening a DNAStream to the DNAStream file
There are two available modes for streaming a DNAStream file, 'r' and 'r+'.   
- 'r' allows read access only and prevents modifications to the DNAStream file (default)
- 'r+' allows read/write access to facilitate modfications to the DNAStream file

The `DNAStream.open(...)` function is used to open the stream to the file and returns a `DNAStream` object to interface with data.

In [2]:
ds = DNAStream.open(path=myfile, mode="r")
print(ds)
ds.close()

DNAStream for Patient patientX with Package Version: 0.1.0b0
File path: '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_oj7bmbg2/temp.h5'
Created by: llweber on host LM4FNLXK2L at 2026-02-05T19:03:06.338336Z
Mode: 'r'
Dataset groups:
	'canonical': 0 tables
	'links': 0 tables
	'measurements': 0 tables
	'provenance': 1 tables
	'registry': 3 tables
	'results': 0 tables



## Using the context manager
It is **reccommended** to stream a DNAStream file with a context manager. This ensure that in the event of errors or exceptions, the stream is automatically closed to print file corruption.

In [3]:
#verbose can be used to print additional messages to the console (default is False)
with DNAStream.open(path=myfile, mode="r", verbose=True) as ds:
    print(ds)


2026-02-05 13:03:06 - INFO - Connection to DNAStream file '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_oj7bmbg2/temp.h5' open.
2026-02-05 13:03:06 - INFO - Connection to DNAStream file '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_oj7bmbg2/temp.h5' closed.


DNAStream for Patient patientX with Package Version: 0.1.0b0
File path: '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_oj7bmbg2/temp.h5'
Created by: llweber on host LM4FNLXK2L at 2026-02-05T19:03:06.338336Z
Mode: 'r'
Dataset groups:
	'canonical': 0 tables
	'links': 0 tables
	'measurements': 0 tables
	'provenance': 1 tables
	'registry': 3 tables
	'results': 0 tables



 `is_connected()` can be used to check the status of the stream.

In [4]:
print(f"DNAStream file '{myfile}' is streaming: {ds.is_connected()}")

DNAStream file '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_oj7bmbg2/temp.h5' is streaming: False


In [5]:
with DNAStream.open(myfile) as ds:
    print(f"DNAStream file '{myfile}' is streaming: {ds.is_connected()}")

DNAStream file '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_oj7bmbg2/temp.h5' is streaming: 1


## DNAStream attributes

`DNAStream` currently includes three built-in registries/provenance attributes and an I/O interface attribute
- `sample`  : sample registry
- `variant` : variant registry
- `snp` : SNP registry
- `log` : File modification provenance
- `io` : I/O interface for DNAStream file



They can be accessed as attributes from a `DNAStream` object or by name via the `registry(name)` or `provenance(name)` function within DNAStream.

Additionally, there are two properties:
- `patient_id` : the id of the patient under study
- `mode` : the current streaming mode of the DNAStream interface

In [6]:
with DNAStream.open(myfile) as ds:
    print(ds.sample)
    print(ds.log)

/registry/sample (Shape: (0,))
/provenance/log (Shape: (4,))


In [7]:
with DNAStream.open(myfile) as ds:
    print(ds.registry("sample"))
    print(ds.provenance("log"))

/registry/sample (Shape: (0,))
/provenance/log (Shape: (4,))


In [8]:
with DNAStream.open(myfile) as ds:
    print(f"DNAStream file for patient: '{ds.patient_id}; is streaming in mode '{ds.mode}'.")

DNAStream file for patient: 'patientX; is streaming in mode 'r'.


The patient id stored in the file can be modified via `set_patient_id()`.

In [9]:
with DNAStream.open(myfile, mode="r+") as ds:
    print(ds)
    ds.set_patient_id("patientY")
    print(ds)

DNAStream for Patient patientX with Package Version: 0.1.0b0
File path: '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_oj7bmbg2/temp.h5'
Created by: llweber on host LM4FNLXK2L at 2026-02-05T19:03:06.338336Z
Mode: 'r+'
Dataset groups:
	'canonical': 0 tables
	'links': 0 tables
	'measurements': 0 tables
	'provenance': 1 tables
	'registry': 3 tables
	'results': 0 tables

DNAStream for Patient patientY with Package Version: 0.1.0b0
File path: '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_oj7bmbg2/temp.h5'
Created by: llweber on host LM4FNLXK2L at 2026-02-05T19:03:06.338336Z
Mode: 'r+'
Dataset groups:
	'canonical': 0 tables
	'links': 0 tables
	'measurements': 0 tables
	'provenance': 1 tables
	'registry': 3 tables
	'results': 0 tables

